# Study 931 — The CEF IPO Hole — the teardown

The abnormal-return table, the vintage-cluster bootstrap, the calendar-time portfolio with its Newey-West *t*, the pseudo-IPO placebo, the seasoned mirror, the era and mapping sweeps, the borrow-swept short, and a live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `d69758d98b1a`); the only live cell is explicitly synthetic.

> 💡 **In plain words:** we are asking whether a brand-new closed-end fund under-performs the very assets it holds, how sure we can be given that only 28 funds and 9 vintages exist, and whether anyone could get paid for knowing it.

In [1]:
R = {'start': '2012-05-25', 'end': '2026-06-30', 'n_days': 3543, 'fp': 'd69758d98b1a', 'n_funds': 28, 'n_vintages': 9, 'offer_ok': 21, 'worst_gap': -7.2, 'm1': -1.47, 'm1_med': -1.12, 'm1_neg': 0.68, 'm1_t': -1.34, 'm3': -5.13, 'm3_med': -4.22, 'm3_neg': 0.79, 'm3_t': -3.94, 'm6': -7.06, 'm6_med': -8.19, 'm6_neg': 0.82, 'm6_t': -4.06, 'm12': -10.26, 'm12_med': -9.45, 'm12_neg': 0.89, 'm12_t': -3.65, 'm12_wilson_lo': 0.73, 'm12_wilson_hi': 0.96, 'cb3_lo': -7.93, 'cb3_hi': -3.47, 'cb6_lo': -10.15, 'cb6_hi': -3.59, 'cb12_lo': -16.39, 'cb12_hi': -3.65, 'cb_clusters': 9, 'ct1_ann': -13.79, 'ct1_t': -1.37, 'ct3_ann': -21.3, 'ct3_t': -2.86, 'ct3_lo': -34.29, 'ct3_hi': -9.23, 'ct6_ann': -11.58, 'ct6_t': -2.12, 'ct12_ann': -5.8, 'ct12_t': -1.57, 'ct12_lo': -12.63, 'ct12_hi': 0.03, 'ct_book': 1.37, 'ct_book_max': 3, 'ct_byfund': -8.37, 'ttd3_med': 27, 'ttd5_med': 34, 'ttd5_q1': 23, 'ttd5_q3': 60, 'ttd5_cross': 27, 'ttd10_med': 97, 'era_e_n': 12, 'era_e': -7.14, 'era_e_neg': 0.83, 'era_e_t': -2.1, 'era_l_n': 16, 'era_l': -12.6, 'era_l_neg': 0.94, 'era_l_t': -2.99, 'map_spy': -16.84, 'map_spy_t': -5.22, 'map_aor': -9.11, 'map_aor_t': -3.08, 'entry_close': -9.55, 'entry_close_t': -3.35, 'plac': 1.9, 'plac_sd': 2.3, 'plac_p05': -1.66, 'plac_p95': 5.8, 'plac_draws': 500, 'seas': 3.02, 'seas_med': 2.85, 'seas_neg': 0.43, 'seas_t': 0.83, 'seas_ct': 2.74, 'seas_ct_t': 1.1, 'sh0': 9.08, 'sh0_t': 3.18, 'sh300': 6.08, 'sh300_t': 2.13, 'sh500': 4.08, 'sh500_t': 1.43, 'sh1000': -0.92, 'sh1000_t': -0.32, 'breakeven_bps': 908, 'breakeven_3m_bps': 1660, 'sh3m_500': 2.9, 'sh3m_500_t': 2.39, 'beta_mean': 0.9, 'beta_med': 0.92, 'beta_min': 0.22, 'beta_max': 1.13, 'car_b1': -8.4, 'car_b1_neg': 0.79, 'car_b1_t': -2.94, 'car_bf': -7.34, 'car_bf_neg': 0.75, 'car_bf_t': -2.56, 'empty_vintages': '2017, 2018', 'syn_planted': -10.0, 'syn_recovered': -9.7, 'syn_t': -6.7, 'syn_null': 0.15, 'syn_null_t': 0.1, 'syn_null_fire': 0, 'syn_lev_b1': 3.3, 'syn_lev_bf': 0.8, 'syn_lev_beta': 1.6}

## Design

- **Sample.** 28 US CEF IPOs, vintages 2012-2022, one asset-class benchmark ETF each (XLK / XLV / IGF / AOR / BKLN / HYG / LQD / PFF).
- **Entry.** The offering price ($20; $25 for the four preferred funds) — an ASSUMPTION, corroborated by the tape for 21/28 funds within 1% (worst -7.2%, RIV, whose Yahoo history starts ~2 weeks late — a bias *against* the effect) and swept against a day-one-close entry.
- **Both legs total return** (`auto_adjust=True`). A price-only comparison would manufacture a fake decay out of an 8-12% distribution rate.
- **One execution lag, once:** the event study's entry *is* the IPO, so the lag lives in the calendar-time portfolio and the mirror trade, both entered at the **t+1** close.
- **The 3-6% load is never subtracted** — it is the mechanism; the tape contains it. Subtracting it would double-count.
- **No excess-of-cash adjustment**, on purpose: every number is a difference of two fully invested legs or a self-financing spread, so the cash rate cancels identically on both sides. BIL is cached and deliberately unread.
- **Universe:** a hand-built **convenience sample** of 28 large, still-listed launches — *not* a census. Several hundred CEFs IPO'd in the window, no free listing-date screen exists, and the **2017, 2018 vintages are empty**. Representativeness cannot be checked from inside this study.
- **Survivorship:** surviving tickers only; BIGZ (2021) left the tape by ETF conversion and XFLT (2017) was dropped for corrupt Yahoo split factors. Worse funds disappear, so the measured hole is biased *toward zero*.

## The event study

In [2]:
print(f"{'h':>4s} {'mean':>8s} {'median':>8s} {'neg':>6s} {'t':>7s}")
for lab in ('1','3','6','12'):
    print(f"{lab+'m':>4s} {R['m'+lab]:+8.2f} {R['m'+lab+'_med']:+8.2f} "
          f"{R['m'+lab+'_neg']:6.0%} {R['m'+lab+'_t']:+7.2f}")
print(f"\n12m share-negative Wilson 95% CI: "
      f"[{R['m12_wilson_lo']:.2f}, {R['m12_wilson_hi']:.2f}]")
print('month 1 is the syndicate stabilisation window; the hole opens in months 2-6')

   h     mean   median    neg       t
  1m    -1.47    -1.12    68%   -1.34
  3m    -5.13    -4.22    79%   -3.94
  6m    -7.06    -8.19    82%   -4.06
 12m   -10.26    -9.45    89%   -3.65

12m share-negative Wilson 95% CI: [0.73, 0.96]
month 1 is the syndicate stabilisation window; the hole opens in months 2-6


## Dependence-robust inference

Twenty-eight overlapping event windows across nine vintages are not 28 independent draws — the six 2021-vintage funds all met the 2022 rate shock together. Two answers.

**(a) Vintage-cluster bootstrap** — resample whole IPO *years* with replacement.

**(b) Calendar-time portfolio** (Fama 1998) — collapse the 28 windows into ONE daily equal-weight long-fund / short-benchmark spread and take its Newey-West *t*, which assumes nothing about cross-fund independence.

> 💡 **In plain words:** (a) asks 'what if we'd drawn different market years?', (b) asks 'what does one honest daily P&L of this idea look like?'

In [3]:
print('vintage-cluster bootstrap (%d clusters):' % R['cb_clusters'])
for lab, lo, hi in (('3m', R['cb3_lo'], R['cb3_hi']), ('6m', R['cb6_lo'], R['cb6_hi']),
                    ('12m', R['cb12_lo'], R['cb12_hi'])):
    print(f"  {lab:>4s}: 95% CI [{lo:+6.2f}%, {hi:+6.2f}%]  -> entirely below zero")
print()
print('calendar-time portfolio (HAC t):')
for lab, ann, t in (('age<=1m', R['ct1_ann'], R['ct1_t']), ('age<=3m', R['ct3_ann'], R['ct3_t']),
                    ('age<=6m', R['ct6_ann'], R['ct6_t']), ('age<=12m', R['ct12_ann'], R['ct12_t'])):
    flag = '  <- clears' if abs(t) >= 2 else ''
    print(f"  {lab:>8s}: {ann:+8.2f}%/yr  HAC t = {t:+.2f}{flag}")
print(f"\nthe 3m book holds {R['ct_book']:.2f} funds on an average day (max {R['ct_book_max']}) "
      f"-> it equal-weights DAYS, not FUNDS, and throws away the cross-sectional averaging;")
print('this is the most conservative test available, not the most powerful one.')

vintage-cluster bootstrap (9 clusters):
    3m: 95% CI [ -7.93%,  -3.47%]  -> entirely below zero
    6m: 95% CI [-10.15%,  -3.59%]  -> entirely below zero
   12m: 95% CI [-16.39%,  -3.65%]  -> entirely below zero

calendar-time portfolio (HAC t):
   age<=1m:   -13.79%/yr  HAC t = -1.37
   age<=3m:   -21.30%/yr  HAC t = -2.86  <- clears
   age<=6m:   -11.58%/yr  HAC t = -2.12  <- clears
  age<=12m:    -5.80%/yr  HAC t = -1.57

the 3m book holds 1.37 funds on an average day (max 3) -> it equal-weights DAYS, not FUNDS, and throws away the cross-sectional averaging;
this is the most conservative test available, not the most powerful one.


The 12-month calendar-time reading (*t* = -1.57) is the honest weak point of this study and is named on the Signal axis: **at twelve months the most conservative test does not clear**, so the 12-month figure is this study's *magnitude* and the 3m/6m figures are its *evidence*. It is an efficiency limit rather than a contradiction — with staggered IPOs the book holds ~1.4 names, equal-weights days rather than funds, and therefore under-weights the crowded (and worst-hit) 2019-2021 cohort; equal-weighting by fund over the same windows gives -8.37%/yr.

> ⚖️ **Audit note.** This test was *tightened* after the build: the book now earns ages 2…window, so it is never credited with the IPO-close → next-close move nobody entering at t+1 could have held. Every horizon moved further from zero (3m −2.69 → -2.86, 6m −1.95 → -2.12, 12m −1.45 → -1.57). The 12-month row fails on either convention.

## Is the hole just leverage?

Subtracting **one** unit of benchmark is an assumption, and closed-end funds are usually 25-40% leveraged: a levered fund launched at the top of its sleeve would print a negative 'abnormal' return with no IPO mechanism at all. So we re-run the event window as a market-model CAR (sum of daily *r*<sub>fund</sub> − β·*r*<sub>bench</sub> from the t+1 close) with each fund's **own β fitted on its seasoned life**, months 12-36.

> ⚠️ **This is a diagnostic, not a rule.** The β comes from data *after* the window it corrects. Nobody standing at the IPO knows it, and it never touches the mirror trade.

In [4]:
print(f"seasoned beta to own benchmark: mean {R['beta_mean']:.2f} "
      f"median {R['beta_med']:.2f}  (range {R['beta_min']:.2f}-{R['beta_max']:.2f})")
print(f"CAR, beta = 1       : {R['car_b1']:+6.2f}%  {R['car_b1_neg']:.0%} negative  "
      f"t = {R['car_b1_t']:+.2f}")
print(f"CAR, beta = fitted  : {R['car_bf']:+6.2f}%  {R['car_bf_neg']:.0%} negative  "
      f"t = {R['car_bf_t']:+.2f}")
print('\n-> mean beta is 0.90, not 1.4: against these benchmarks the funds are not')
print('   high-beta, and adjusting for beta removes ~a fifth of the hole and none')
print('   of its significance. The hole is not leverage.')

seasoned beta to own benchmark: mean 0.90 median 0.92  (range 0.22-1.13)
CAR, beta = 1       :  -8.40%  79% negative  t = -2.94
CAR, beta = fitted  :  -7.34%  75% negative  t = -2.56

-> mean beta is 0.90, not 1.4: against these benchmarks the funds are not
   high-beta, and adjusting for beta removes ~a fifth of the hole and none
   of its significance. The hole is not leverage.


## Is it the IPO window, or the funds?

The placebo re-runs the identical 12-month estimator on the identical funds and benchmarks from **random seasoned start dates**. If these were simply bad assets, it would reproduce the hole.

In [5]:
print(f"real IPO window       : {R['m12']:+6.2f}%  (t = {R['m12_t']:+.2f})")
print(f"pseudo-IPO placebo    : {R['plac']:+6.2f}%  (sd {R['plac_sd']:.2f}%, "
      f"5-95pct [{R['plac_p05']:+.2f}%, {R['plac_p95']:+.2f}%], {R['plac_draws']} draws)")
print(f"same funds, months 12-36: {R['seas']:+6.2f}%  (t = {R['seas_t']:+.2f}, "
      f"{R['seas_neg']:.0%} negative)")
print(f"  calendar-time version : {R['seas_ct']:+6.2f}%/yr (HAC t = {R['seas_ct_t']:+.2f})")
print('\n-> the damage is concentrated in the IPO window and stops once seasoned.')

real IPO window       : -10.26%  (t = -3.65)
pseudo-IPO placebo    :  +1.90%  (sd 2.30%, 5-95pct [-1.66%, +5.80%], 500 draws)
same funds, months 12-36:  +3.02%  (t = +0.83, 43% negative)
  calendar-time version :  +2.74%/yr (HAC t = +1.10)

-> the damage is concentrated in the IPO window and stops once seasoned.


## Robustness — era cut, benchmark mapping, entry convention

In [6]:
print(f"vintages 2012-2016 (n={R['era_e_n']}): {R['era_e']:+6.2f}%  "
      f"{R['era_e_neg']:.0%} negative  t={R['era_e_t']:+.2f}")
print(f"vintages 2019-2022 (n={R['era_l_n']}): {R['era_l']:+6.2f}%  "
      f"{R['era_l_neg']:.0%} negative  t={R['era_l_t']:+.2f}")
print()
print(f"benchmark = asset class : {R['m12']:+6.2f}%  t={R['m12_t']:+.2f}  (headline)")
print(f"benchmark = SPY         : {R['map_spy']:+6.2f}%  t={R['map_spy_t']:+.2f}")
print(f"benchmark = AOR (60/40) : {R['map_aor']:+6.2f}%  t={R['map_aor_t']:+.2f}")
print()
print(f"entry = offering price  : {R['m12']:+6.2f}%  t={R['m12_t']:+.2f}")
print(f"entry = day-one close   : {R['entry_close']:+6.2f}%  t={R['entry_close_t']:+.2f}")
print('\n-> entering at the day-one close recovers only 0.7pp of the 10.3pp hole:')
print('   the damage is post-listing drift, not the day-one gap.')

vintages 2012-2016 (n=12):  -7.14%  83% negative  t=-2.10
vintages 2019-2022 (n=16): -12.60%  94% negative  t=-2.99

benchmark = asset class : -10.26%  t=-3.65  (headline)
benchmark = SPY         : -16.84%  t=-5.22
benchmark = AOR (60/40) :  -9.11%  t=-3.08

entry = offering price  : -10.26%  t=-3.65
entry = day-one close   :  -9.55%  t=-3.35

-> entering at the day-one close recovers only 0.7pp of the 10.3pp hole:
   the damage is post-listing drift, not the day-one gap.


## Time-to-discount PROXY

In [7]:
for thr, med in ((3, R['ttd3_med']), (5, R['ttd5_med']), (10, R['ttd10_med'])):
    print(f"first day {thr:2d}% behind the asset class: median {med:3d} trading days")
print(f"  (-5% threshold: {R['ttd5_cross']}/{R['n_funds']} funds cross within 2 years, "
      f"IQR [{R['ttd5_q1']}, {R['ttd5_q3']}] days)")
print('\nPROXY: daily NAV is not on the free tape, so this conflates the price/NAV')
print('discount with fund-vs-ETF tracking difference. The timing pattern is the read.')

first day  3% behind the asset class: median  27 trading days
first day  5% behind the asset class: median  34 trading days
first day 10% behind the asset class: median  97 trading days
  (-5% threshold: 27/28 funds cross within 2 years, IQR [23, 60] days)

PROXY: daily NAV is not on the free tape, so this conflates the price/NAV
discount with fund-vs-ETF tracking difference. The timing pattern is the read.


## Could you trade it? Short the new issue, long the benchmark

Equal notional, opened at the **t+1** close, 20 bps one-way × NAV on each of four legs (two on entry, two on exit), borrow charged pro-rata on the short leg.

> 💡 **In plain words:** the edge is real but it lives on the short side, and new closed-end funds are among the least borrowable securities in existence.

In [8]:
print(f"{'borrow':>10s} {'net 12m':>9s} {'t':>7s}")
for bo, net, t in ((0, R['sh0'], R['sh0_t']), (300, R['sh300'], R['sh300_t']),
                   (500, R['sh500'], R['sh500_t']), (1000, R['sh1000'], R['sh1000_t'])):
    print(f"{bo:8d}bp {net:+9.2f}% {t:+7.2f}")
print(f"\nbreak-even borrow: ~{R['breakeven_bps']} bps/yr at 12m, "
      f"~{R['breakeven_3m_bps']} bps/yr at 3m")
print(f"3m version at 500bp borrow: {R['sh3m_500']:+.2f}% (t = {R['sh3m_500_t']:+.2f})")
print('\nA CEF weeks after its IPO has essentially no lendable float: the whole issue')
print('sits in the retail accounts the syndicate placed it into. Paper trade.')

    borrow   net 12m       t
       0bp     +9.08%   +3.18
     300bp     +6.08%   +2.13
     500bp     +4.08%   +1.43
    1000bp     -0.92%   -0.32

break-even borrow: ~908 bps/yr at 12m, ~1660 bps/yr at 3m
3m version at 500bp borrow: +2.90% (t = +2.39)

A CEF weeks after its IPO has essentially no lendable float: the whole issue
sits in the retail accounts the syndicate placed it into. Paper trade.


## Live synthetic control — the estimator is unbiased *(synthetic, not the real tape)*

Planted panel: 28 staggered synthetic IPOs each carrying a −10% first-year slide, which the estimator must recover. Null panel: the same generator with nothing planted, on which it must stay quiet across seeds. **No real fund appears below.**

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from cef_ipo import data, strategy as st
planted, truth = data.synthetic_panel(signal_strength=1.0, seed=931)
p = st.synthetic_detect(planted)
print('SYNTHETIC planted %+.1f%%: recovered %+.2f%% (t = %+.2f, %.0f%% of funds negative)'
      % (-truth['planted_slide']*100, p['mean_abn_pct'], p['tstat'], p['share_negative']*100))
ts = np.array([st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=931+s)[0])['tstat']
               for s in range(8)])
print('SYNTHETIC null x8 seeds : t mean %+.2f (sd %.2f), |t|>=2 in %d/8'
      % (ts.mean(), ts.std(ddof=1), int((np.abs(ts) >= 2).sum())))
half = st.synthetic_detect(data.synthetic_panel(signal_strength=0.5, seed=931)[0])
print('SYNTHETIC dose-response : half-strength %+.2f%% vs full %+.2f%%'
      % (half['mean_abn_pct'], p['mean_abn_pct']))
print('SYNTHETIC levered null  : beta %.1f planted and NO hole -> beta=1 CAR %+.2f%% '
      '(biased), beta-fitted CAR %+.2f%% (unbiased)'
      % (R['syn_lev_beta'], R['syn_lev_b1'], R['syn_lev_bf']))

SYNTHETIC planted -10.0%: recovered -9.70% (t = -6.70, 89% of funds negative)


SYNTHETIC null x8 seeds : t mean +0.75 (sd 0.40), |t|>=2 in 0/8


SYNTHETIC dose-response : half-strength -4.89% vs full -9.70%
SYNTHETIC levered null  : beta 1.6 planted and NO hole -> beta=1 CAR +3.30% (biased), beta-fitted CAR +0.80% (unbiased)


## Verdict

- **Signal — Real.** The stamp rests on the horizons where *every* estimator agrees: -5.13% by month three (one-sample *t* = **-3.94**, calendar-time HAC *t* = **-2.86**, vintage-cluster CI [-7.93%, -3.47%]) and -7.06% by month six (*t* = -4.06, HAC *t* = **-2.12**). The twelve-month magnitude is -10.26% (89% of funds negative, Wilson [0.73, 0.96]). Both vintage halves clear |*t*| ≥ 2 alone (-2.10 / -2.99); all three benchmark mappings reproduce it (-16.84% to -9.11%); each fund's own fitted beta removes only a fifth of it (-8.40% → -7.34%, *t* = -2.56); the pseudo-IPO placebo reads +1.90% and the seasoned window +3.02% (*t* = +0.83), so the damage is IPO-specific. The synthetic control recovers a planted −10% as -9.70% (*t* = -6.70) and fires on 0/8 null seeds. **Named on this axis:** at **12 months the calendar-time HAC *t* is -1.57 and does not clear** on a ~1.4-name book — the 12m number is the magnitude, not the evidence; the **universe is a hand-built convenience sample of 28 large launches, not a census** (2017, 2018 contribute nothing); survivorship (surviving tickers only, BIGZ converted away, XFLT dropped for corrupt splits) biases the hole toward zero; 28 funds across 9 vintages; the offering price, the benchmark mapping and beta = 1 are assumptions, all corroborated and swept.
- **Tradability — Mirage.** +9.08% over twelve months at zero borrow, but break-even borrow is ~908 bps/yr and a newly issued CEF has no lendable float; there is no long expression of a hole. The residual is an abstention rule — do not subscribe, buy seasoned — which avoids a loss and banks nothing.